# Glüten — Gemma 4 E4B Marsh classifier (Unsloth QLoRA, v2)

**Task:** Fine-tune `google/gemma-4-E4B-it` (the official Google base) on IBDColEpi HE patches to classify a proxy Marsh grade.

**Why a v2:** the v1 notebook trained against `unsloth/gemma-4-E4B-it-unsloth-bnb-4bit`, which is a pre-quantized 4-bit variant. The resulting LoRA only loads cleanly on top of that exact 4-bit base, and the bitsandbytes runtime path has a regression with multimodal Gemma in current `transformers`. v2 trains against the official non-quantized Google base, so the resulting fp16 LoRA loads cleanly on **any** `google/gemma-4-E4B-it` deployment (Modal, Vertex, local 32GB+) with no bitsandbytes anywhere in the inference path.

**What stays the same:** Unsloth as the training framework (still targets the Unsloth special-technology prize), QLoRA approach, dataset, labels, eval split, LoRA hyperparameters. The only change is the base-model name and `load_in_4bit=False`.

**Honest framing — read this first.** IBDColEpi ships with pixel-level epithelium segmentation masks but **no Marsh / villous-atrophy / IEL annotations**. The labels trained below are weak-supervision proxies derived from epithelium mask coverage per patch (lower coverage ≈ more atrophy). They are methodologically principled but **not pathologist-validated Marsh scores**. The hackathon writeup and the `/api/medgemma/marsh` endpoint surface this caveat explicitly.

**Target prize track:** Unsloth special technology (training framework unchanged from v1).

**Runtime:** Kaggle free T4 (~30-45 min, slightly slower than v1 because we're not in 4-bit). Enable *Accelerator → GPU T4* and *Internet → On*.

**Attached data:** the private Kaggle dataset `gluten-ibdcolepi-sample` (uploaded from `data/structural/processed/patch-dataset-HE-sampled.zip` + `marsh_pseudo_labels-sampled.csv`).

**Setup quirks already accounted for in this notebook (don't undo them):**
- Pillow is left at Kaggle's pre-installed 12.2.0 — Unsloth installed with `--no-deps` so it can't downgrade Pillow and break the `_imaging` C extension.
- Vision tower is **frozen** (`finetune_vision_layers=False`). We only LoRA-tune the language head.
- `google/gemma-4-E4B-it` is gated — your Kaggle `HF_TOKEN` secret must come from a HuggingFace account that has clicked Acknowledge license on the model page.

In [ ]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q --upgrade transformers
!pip install -q bitsandbytes trl peft accelerate tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import transformers; print('Transformers:', transformers.__version__)
import PIL; print('Pillow:', PIL.__version__)

## 1 · Install Unsloth + deps

All `--no-deps` so we don't fight Kaggle's base Pillow / transformers / datasets versions. If this cell errors, do **Run → Factory reset** and rerun from cell 1.

In [ ]:
%%capture
!pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q --no-deps cut-cross-entropy
!pip install -q tifffile

import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
import PIL; print('Pillow:', PIL.__version__)  # should print 12.2.0

## 2 · Build pseudo-Marsh labels from epithelium masks

For each HE patch we compute `epi_frac = (mask > 0).mean()`, then quantile-bin into Marsh-0 / Marsh-1 / Marsh-3a / Marsh-3b. Marsh-2 is omitted (under-represented in literature; molecular layer dataset GSE164883 also skips it). ~30 s on Kaggle disk.

In [ ]:
import pandas as pd, numpy as np
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/faithogun/gluten-ibdcolepi-sample')
PATCHES = DATA_ROOT / 'patch-dataset-HE-sampled'
SEED_CSV = DATA_ROOT / 'marsh_pseudo_labels-sampled.csv'

seed = pd.read_csv(SEED_CSV)
print(seed.shape, seed['split'].value_counts())

def read_mask(label_path):
    return np.array(Image.open(PATCHES / label_path))
def read_image(image_path):
    return Image.open(PATCHES / image_path).convert('RGB')

seed['epi_frac'] = [float((read_mask(r['label_path']) > 0).mean())
                    for _, r in tqdm(seed.iterrows(), total=len(seed))]

q = seed['epi_frac'].quantile([0.25, 0.5, 0.75]).values
def bin_fn(f):
    if f >= q[2]: return 'Marsh-0'
    if f >= q[1]: return 'Marsh-1'
    if f >= q[0]: return 'Marsh-3a'
    return 'Marsh-3b'
seed['marsh_bin'] = seed['epi_frac'].apply(bin_fn)
print(seed['marsh_bin'].value_counts())
seed.to_csv('/kaggle/working/marsh_pseudo_labels_scored.csv', index=False)

## 3 · Load Gemma 4 E4B in 4-bit (fits T4), attach LoRA to the language head only

**Why 4-bit at training time but fp16 at deploy time.** Gemma 4 E4B is ~16 GB in fp16 and does not fit on a free T4 (14.56 GB). Loading in 4-bit drops it to ~5 GB, leaving room for gradients and optimizer state. After training, Unsloth's `save_pretrained_merged(save_method='merged_16bit')` **dequantizes the 4-bit base and folds the LoRA in**, producing a clean fp16 single-model checkpoint with no bitsandbytes anywhere in the inference path. That sidesteps the bnb+multimodal-Gemma regression in transformers 5.x that broke v1 deployment.

`finetune_vision_layers=False` is the load-bearing line. We freeze the SigLIP vision encoder and only LoRA-train the language part.


In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'google/gemma-4-E4B-it',
    load_in_4bit=True,            # 4-bit at training time so we fit on T4 (~5 GB base)
    use_gradient_checkpointing='unsloth',
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', random_state=42,
)
print('Loaded Gemma 4 E4B (4-bit official base), vision tower frozen, LoRA on language head.')


## 4 · Build the image→label instruction dataset

In [ ]:
from datasets import Dataset

INSTRUCTION = (
    'You are a histopathology assistant. Classify the Marsh grade of this HE-stained '
    'intestinal biopsy patch. Respond with exactly one of: Marsh-0, Marsh-1, Marsh-3a, Marsh-3b.'
)

def load_patch(image_path):
    return read_image(image_path).resize((224, 224))

def to_example(row):
    img = load_patch(row['image_path'])
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': INSTRUCTION},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': row['marsh_bin']}]},
        ]
    }

train_df = seed[seed['split'] == 'Trainset'].sample(min(2000, (seed['split'] == 'Trainset').sum()), random_state=42)
val_df = seed[seed['split'] == 'Validationset']

train_ds = Dataset.from_list([to_example(r) for _, r in train_df.iterrows()])
val_ds = Dataset.from_list([to_example(r) for _, r in val_df.iterrows()])
print(train_ds, val_ds)

## 5 · Train

Mixed precision (fp16 on T4). Expect ~30 min for 250 steps. If it OOMs, drop `per_device_train_batch_size` to 1 and bump `gradient_accumulation_steps` to 8.

**Don't sit and wait** — use *Save Version → Save & Run All (Commit)* to run this in the background once the notebook is correct.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_torch',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/kaggle/working/gemma4-marsh-lora',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=1024,
    ),
)
trainer.train()

## 6 · Evaluate on the held-out Testset split

In [ ]:
FastVisionModel.for_inference(model)
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained('google/gemma-4-E4B-it')

test_df = seed[seed['split'] == 'Testset'].sample(min(400, (seed['split'] == 'Testset').sum()), random_state=42)
preds, gold = [], []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img = load_patch(row['image_path'])
    msgs = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': INSTRUCTION}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(**inputs, max_new_tokens=8, temperature=0, do_sample=False)
    txt = processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    preds.append(txt.split()[0] if txt else '')
    gold.append(row['marsh_bin'])

print(classification_report(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b'], zero_division=0))
print(confusion_matrix(gold, preds, labels=['Marsh-0','Marsh-1','Marsh-3a','Marsh-3b']))

## 7 · Save **merged fp16** model (not just the adapter)

This is the load-bearing fix vs v1. Instead of saving only the LoRA (which forces the deploy environment to also load the 4-bit base and bitsandbytes), we use Unsloth's `save_pretrained_merged(save_method='merged_16bit')` to:

1. Dequantize the 4-bit base weights back to fp16
2. Fold the LoRA delta into them
3. Write a single ~15 GB fp16 checkpoint

That checkpoint loads on Modal (or anywhere) with vanilla `AutoModelForImageTextToText.from_pretrained(..., dtype=torch.bfloat16)` and **no bnb, no PEFT**. The bnb-multimodal regression in transformers 5.x cannot bite us anymore.

**Disk-space note.** Unsloth's merge writes the full file *and* tries to pre-shard it for upload. On Kaggle's 20 GB working volume this overflows. The cell below cleans up the temp shards immediately after merge so the HF push in §8 has clean inputs.


In [ ]:
import os, glob
from pathlib import Path

OUT = '/kaggle/working/gemma4-marsh-merged-fp16'

# Unsloth's save_pretrained_merged dequantizes the 4-bit base and folds the
# LoRA into it. Result: a clean fp16 single-model checkpoint, no bnb at deploy.
model.save_pretrained_merged(
    OUT,
    tokenizer,
    save_method='merged_16bit',
)

# Unsloth tries to shard the merged file into smaller safetensors for upload.
# On Kaggle's 20 GB working volume that step runs out of disk (15 GB main
# file + ~4 GB of half-written temp shards). The merge itself still completes
# successfully. Clean up the temp_split_* files immediately so the HF push
# in the next cell doesn't try to read corrupted half-files.
for f in glob.glob(f'{OUT}/temp_split_*.safetensors'):
    os.remove(f)
    print('removed', f)

!ls -lh /kaggle/working/gemma4-marsh-merged-fp16/
!df -h /kaggle/working
print('Merged fp16 checkpoint saved + temp shards cleaned.')


## 8 · Push the merged checkpoint to HuggingFace (optional but recommended)

Pushing from Kaggle is much faster than downloading 16 GB locally and re-uploading. The Modal sidecar pulls from HF at boot, so this is the simplest path to `live` mode.

**Prereqs:**
- Kaggle secret `HF_TOKEN` is set (Settings → Secrets) from a HuggingFace account that has clicked Acknowledge on the Gemma 4 license.
- The target repo `faith-ogun/gluten-gemma4-marsh-merged` will be created private on first push.


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, create_repo

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
REPO = 'faith-ogun/gluten-gemma4-marsh-merged'

create_repo(REPO, repo_type='model', private=True, exist_ok=True, token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path='/kaggle/working/gemma4-marsh-merged-fp16',
    repo_id=REPO,
    repo_type='model',
    ignore_patterns=['temp_split_*', '*.tmp'],
    commit_message='Gemma 4 E4B + Marsh LoRA merged fp16 (v2, no bnb at deploy)',
)
print(f'Pushed: https://huggingface.co/{REPO}')
